# RootBot: прямокутники з CVAT -> контури фігурок

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Встав Kaggle-токен, потім вибери zip-и з CVAT.
4. В кінці завантажиться `outlines.zip`. Червоні контури в `previews/` - там SAM не впорався.

In [ ]:
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Kaggle token: ')
!pip install -q -U ultralytics kaggle
!kaggle datasets download natalyakoz/rootbot-frames -p /content/frames --unzip -q

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files
for name in files.upload():
    zipfile.ZipFile(name).extractall(Path('/content/labels') / Path(name).stem)

In [ ]:
import csv, re, shutil
from pathlib import Path
import cv2, numpy as np
from ultralytics import SAM

FRAMES, LABELS, OUT = Path('/content/frames'), Path('/content/labels'), Path('/content/out')
for d in ('dataset/images', 'dataset/labels', 'previews'):
    (OUT / d).mkdir(parents=True, exist_ok=True)
model = SAM('sam2.1_b.pt')
frames = {p.stem: p for p in FRAMES.rglob('*.jpg')}

def read_boxes(txt, W, H):
    boxes = []
    for line in txt.read_text().splitlines():
        c, x, y, w, h = line.split()
        x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
        boxes.append((int(c), max(0, int(x - w / 2)), max(0, int(y - h / 2)),
                      min(W, int(x + w / 2)), min(H, int(y + h / 2))))
    return boxes

def box_mask(img, x1, y1, x2, y2):
    # SAM on a crop around the box: the piece is larger, so the edges are sharper
    H, W = img.shape[:2]
    pad = max(x2 - x1, y2 - y1, 64)
    X1, Y1, X2, Y2 = max(0, x1 - pad), max(0, y1 - pad), min(W, x2 + pad), min(H, y2 + pad)
    crop = np.ascontiguousarray(img[Y1:Y2, X1:X2])
    r = model(crop, bboxes=[[x1 - X1, y1 - Y1, x2 - X1, y2 - Y1]], verbose=False)[0]
    mask = np.zeros((H, W), bool)
    if r.masks is not None:
        mask[Y1:Y2, X1:X2] = r.masks.data[0].cpu().numpy().astype(bool)
    limit = np.zeros((H, W), bool)
    limit[max(0, y1 - 4):y2 + 4, max(0, x1 - 4):x2 + 4] = True  # the box is tight: no leaks outside it
    return mask & limit

def split_overlaps(masks, boxes):
    # a pixel claimed by two pieces goes to the piece whose box centre is nearer
    ys, xs = np.nonzero(np.sum(masks, axis=0) > 1)
    for y, x in zip(ys, xs):
        owners = [i for i, m in enumerate(masks) if m[y, x]]
        best = min(owners, key=lambda i: (x - (boxes[i][1] + boxes[i][3]) / 2) ** 2 + (y - (boxes[i][2] + boxes[i][4]) / 2) ** 2)
        for i in owners:
            masks[i][y, x] = i == best

def polygon(mask, box):
    # largest connected part as a polygon; the rectangle itself if SAM failed
    _, x1, y1, x2, y2 = box
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cs:
        c = max(cs, key=cv2.contourArea)
        if cv2.contourArea(c) >= 0.25 * (x2 - x1) * (y2 - y1):
            p = cv2.approxPolyDP(c, 1.0, True).reshape(-1, 2)
            if len(p) >= 3:
                return p, False
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]]), True

rows, names = [], None
for txt in sorted(LABELS.rglob('labels/train/*.txt')):
    names = names or txt.parents[2] / 'data.yaml'
    img = cv2.imread(str(frames[txt.stem]))
    H, W = img.shape[:2]
    boxes = read_boxes(txt, W, H)
    masks = [box_mask(img, *b[1:]) for b in boxes]
    if masks:
        split_overlaps(masks, boxes)
    lines, failed, view = [], 0, img.copy()
    for b, m in zip(boxes, masks):
        p, bad = polygon(m, b)
        failed += bad
        lines.append(f'{b[0]} ' + ' '.join(f'{x / W:.6f} {y / H:.6f}' for x, y in p))
        cv2.polylines(view, [p.reshape(-1, 1, 2)], True, (0, 0, 255) if bad else (0, 255, 255), 2)
    (OUT / 'dataset/labels' / txt.name).write_text('\n'.join(lines) + '\n')
    shutil.copy(frames[txt.stem], OUT / 'dataset/images')
    cv2.imwrite(str(OUT / 'previews' / f'{txt.stem}.jpg'), view)
    rows.append((txt.stem, len(boxes), failed))
    print(*rows[-1])

classes = re.findall(r'^\s+\d+:\s*(\S+)', names.read_text(), re.M)
(OUT / 'dataset/data.yaml').write_text('path: .\ntrain: images\nval: images\nnames:\n' + ''.join(f'  {i}: {n}\n' for i, n in enumerate(classes)))
with open(OUT / 'report.csv', 'w', newline='') as f:
    csv.writer(f).writerows([('frame', 'boxes', 'sam_failed')] + rows)
print('frames', len(rows), 'boxes', sum(r[1] for r in rows), 'sam_failed', sum(r[2] for r in rows))

In [ ]:
!cd /content/out && zip -qr /content/outlines.zip dataset previews report.csv
from google.colab import files
files.download('/content/outlines.zip')